In [7]:
pip install pyspark

Note: you may need to restart the kernel to use updated packages.


In [8]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import FloatType, IntegerType, StringType
from pyspark.sql import Window
import re, os, json

In [9]:
import os
from pyspark.sql import SparkSession

# Since you're in /notebooks, go up one level to reach /data
DATA_DIR = "../data"
PITSTOPS_XLSX = os.path.join(DATA_DIR, "pitstop.xlsx")
LAP_TIMES_XLSX = os.path.join(DATA_DIR, "lap_times.xlsx")
SAFETYCAR_XLSX = os.path.join(DATA_DIR, "safety_cars.xlsx")
REDFLAG_XLSX = os.path.join(DATA_DIR, "red_flags.xlsx")
OUTPUT_PARQUET = "output/f1_full_engineered.parquet"
OUTPUT_CSV = "output/f1_full_engineered_csv"
ANONYMIZE_DRIVERS = True

In [10]:
print("\nChecking file existence with corrected paths:")
files = {
    "Pitstops": PITSTOPS_XLSX,
    "Lap Times": LAP_TIMES_XLSX, 
    "Safety Car": SAFETYCAR_XLSX,
    "Red Flag": REDFLAG_XLSX
}

for name, path in files.items():
    exists = os.path.exists(path)
    print(f"{'yes' if exists else 'no'} {name}: {os.path.basename(path)}")


Checking file existence with corrected paths:
yes Pitstops: pitstop.xlsx
yes Lap Times: lap_times.xlsx
yes Safety Car: safety_cars.xlsx
yes Red Flag: red_flags.xlsx


In [11]:
conda install -c conda-forge openjdk=17

Jupyter detected...
2 channel Terms of Service accepted
doneieving notices: - 
Channels:
 - conda-forge
 - defaults
Platform: osx-arm64
doneng package metadata (repodata.json): 
doneing environment: / 


==> WARNING: A newer version of conda exists. <==
    current version: 25.7.0
    latest version: 25.9.1

Please update conda by running

    $ conda update -n base -c conda-forge conda



# All requested packages already installed.


Note: you may need to restart the kernel to use updated packages.


In [12]:
spark = (SparkSession.builder
         .appName("F1FullPipeline")
         .master("local[*]")
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/11/17 23:00:19 WARN Utils: Your hostname, Divyanshs-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 10.33.74.20 instead (on interface en0)
25/11/17 23:00:19 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/17 23:00:20 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [13]:
def normalize_text(s):
    if s is None: return None
    s = re.sub(r"[^\x00-\x7F]+","",str(s))
    s = re.sub(r"\s+"," ", s)
    return s.strip()
normalize_udf = udf(normalize_text, StringType())

def parse_laptime_to_seconds(t):
    if t is None: return None
    t = str(t).strip()
    if t=="": return None
    try: return float(t)
    except: pass
    m = re.match(r"(?:(\d+):)?(\d+)(?:[:\.](\d+))?$", t)
    if not m: return None
    g = m.groups()
    minutes = int(g[0]) if g[0] else 0
    seconds = int(g[1])
    ms = float("0."+g[2]) if g[2] else 0
    return minutes*60 + seconds + ms
parse_laptime_udf = udf(parse_laptime_to_seconds, FloatType())

In [17]:
# Stop existing Spark session
spark.stop()

# Restart Spark with the Excel package
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("F1ExcelPipeline") \
    .config("spark.jars.packages", "com.crealytics:spark-excel_2.12:0.18.3") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 57029)
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/socketserver.py", line 318, in _handle_request_noblock
    self.process_request(request, client_address)
    ~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.13/socketserver.py", line 349, in process_request
    self.finish_request(request, client_address)
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.13/socketserver.py", line 362, in finish_request
    self.RequestHandlerClass(request, client_address, self)
    ~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.13/socketserver.py", line 766, in __init__
    self.handle()
    ~~~~~~~~~~~^^
  File "/opt/anaconda3/lib/python3.13/site-packages/pyspark/accumulators.py", line 299, in handle
    poll(accum_updates)
    ~~~~^^^^^^^^^^^^^^^
  File "/o

In [22]:
def excel_to_spark_via_pandas(excel_path):
    """Convert Excel to Spark DataFrame using pandas as bridge"""
    try:
        if os.path.exists(excel_path):
            print(f" Loading via pandas: {os.path.basename(excel_path)}")
            
            import pandas as pd
            # Read with pandas
            pandas_df = pd.read_excel(excel_path)
            
            # Convert datetime.time objects to strings for Spark compatibility
            for column in pandas_df.columns:
                if pandas_df[column].dtype == 'object':
                    # Check if the column contains datetime.time objects
                    sample_value = pandas_df[column].dropna().iloc[0] if not pandas_df[column].dropna().empty else None
                    if hasattr(sample_value, 'strftime'):
                        # Convert time objects to string
                        pandas_df[column] = pandas_df[column].apply(
                            lambda x: x.strftime('%H:%M:%S') if pd.notna(x) and hasattr(x, 'strftime') else x
                        )
            
            # Convert to Spark DataFrame
            spark_df = spark.createDataFrame(pandas_df)
            
            print(f" Success: {spark_df.count()} rows, {len(spark_df.columns)} columns")
            return spark_df
        else:
            print(f" File not found: {os.path.basename(excel_path)}")
            return None
    except Exception as e:
        print(f" Error: {e}")
        # Try alternative approach
        return excel_to_spark_alternative(excel_path)

def excel_to_spark_alternative(excel_path):
    """Alternative method for problematic Excel files"""
    try:
        import pandas as pd
        # Read with pandas and convert all columns to string first
        pandas_df = pd.read_excel(excel_path)
        
        # Convert all columns to string to avoid serialization issues
        for col in pandas_df.columns:
            pandas_df[col] = pandas_df[col].astype(str)
        
        spark_df = spark.createDataFrame(pandas_df)
        print(f" Success (alternative method): {spark_df.count()} rows, {len(spark_df.columns)} columns")
        return spark_df
    except Exception as e:
        print(f" Alternative method also failed: {e}")
        return None

# Load files again with the fixed function
print("Loading files with fixed function...")
pit = excel_to_spark_via_pandas(PITSTOPS_XLSX)
laps = excel_to_spark_via_pandas(LAP_TIMES_XLSX)
safety = excel_to_spark_via_pandas(SAFETYCAR_XLSX)
redflag = excel_to_spark_via_pandas(REDFLAG_XLSX)

Loading files with fixed function...
 Loading via pandas: pitstop.xlsx
 Success: 7374 rows, 30 columns
 Loading via pandas: lap_times.xlsx


25/11/17 23:07:14 WARN TaskSetManager: Stage 24 contains a task of very large size (1626 KiB). The maximum recommended task size is 1000 KiB.


 Success: 589081 rows, 6 columns
 Loading via pandas: safety_cars.xlsx
 Success: 362 rows, 5 columns
 Loading via pandas: red_flags.xlsx
 Success: 98 rows, 5 columns


In [23]:
from pyspark.sql.functions import col, struct, collect_list, avg, sum, first, when, broadcast

if pit is not None:
    pit_clean = (pit.withColumn("race_norm", normalize_udf(col("Race Name")))
                 .withColumn("driver_norm", normalize_udf(col("Driver")))
                 .withColumn("pit_time_s", col("Pit_Time").cast(FloatType()))
                 .withColumn("stint_num", col("Stint").cast(IntegerType()))
                 .withColumn("Stint_Length_num", col("Stint Length").cast(IntegerType()))
                )
    stint_struct = struct(col("stint_num").alias("stint"),
                          col("Tire Compound").alias("tire"),
                          col("Stint_Length_num").alias("length"),
                          col("pit_time_s").alias("pit_time"))
    group_cols = ["Season","Round","race_norm","driver_norm","Constructor"]
    pit_merged = pit_clean.groupBy(*group_cols).agg(
        collect_list(stint_struct).alias("stints"),
        avg("pit_time_s").alias("avg_pit_time"),
        sum("pit_time_s").alias("total_pit_time"),
        first("Circuit").alias("Circuit"),
        first("Country").alias("Country")
    )

# Fixed indentation for safety and redflag
if safety is not None:
    safety = safety.withColumn("race_norm", normalize_udf(col("Race")))
if redflag is not None:
    redflag = redflag.withColumn("race_norm", normalize_udf(col("Race")))

if pit_merged is not None:
    from pyspark.sql.functions import broadcast
    joined = pit_merged
    if safety is not None:
        joined = joined.join(broadcast(safety), "race_norm", "left")
    if redflag is not None:
        joined = joined.join(broadcast(redflag), "race_norm", "left")
    joined = joined.withColumn("had_safety_car", when(col("FullLaps").isNotNull(), True).otherwise(False)) \
                   .withColumn("had_red_flag", when(col("Lap").isNotNull(), True).otherwise(False))

In [24]:
def flatten_stints(stints):
    if stints is None: return ""
    s_sorted = sorted([s.asDict() for s in stints if s is not None], key=lambda x:x.get("stint",0))
    return ",".join([f"{s['stint']}:{s['tire']}({s['length']})" for s in s_sorted])
flatten_udf = udf(flatten_stints,StringType())
if joined is not None:
    joined = joined.withColumn("stints_serial", flatten_udf(col("stints")))


In [25]:
if laps is not None:
    if "milliseconds" in laps.columns:
        laps = laps.withColumn("lap_time_s", col("milliseconds")/1000)
    elif "time" in laps.columns:
        laps = laps.withColumn("lap_time_s", parse_laptime_udf(col("time")))
    for c in ["raceId","driverId","lap"]:
        if c in laps.columns: laps = laps.withColumn(c, col(c).cast(IntegerType()))
    if "driverName" in laps.columns: laps = laps.withColumn("driver_norm", normalize_udf(col("driverName")))
    w = Window.partitionBy("raceId","driverId").orderBy("lap").rowsBetween(Window.unboundedPreceding,0)
    laps = laps.withColumn("cum_time_s", sum("lap_time_s").over(w))
    # gaps
    w2 = Window.partitionBy("raceId","lap")
    laps = laps.withColumn("leader_cum_time", min("cum_time_s").over(w2))
    laps = laps.withColumn("gap_to_leader", col("cum_time_s")-col("leader_cum_time"))
    w_rank = Window.partitionBy("raceId","lap").orderBy(col("cum_time_s"))
    from pyspark.sql.functions import row_number, lead, lag
    laps = laps.withColumn("pos_inlap", row_number().over(w_rank))
    laps = laps.withColumn("next_cum_time", lead("cum_time_s").over(w_rank))
    laps = laps.withColumn("prev_cum_time", lag("cum_time_s").over(w_rank))
    laps = laps.withColumn("gap_to_next", when(col("next_cum_time").isNotNull(), col("next_cum_time")-col("cum_time_s")))
    laps = laps.withColumn("gap_to_prev", when(col("prev_cum_time").isNotNull(), col("cum_time_s")-col("prev_cum_time")))
    laps = laps.drop("leader_cum_time","next_cum_time","prev_cum_time")


In [33]:
from pyspark.sql.functions import col, udf, avg, when, regexp_replace, first
from pyspark.sql.types import StringType, FloatType

# Define the normalization UDF
def normalize_text(text):
    if text is None:
        return None
    return str(text).lower().strip().replace(" ", "_")

normalize_udf = udf(normalize_text, StringType())

# Apply normalization to pit DataFrame with safe casting
if pit is not None:
    pit = pit.withColumn("pit_time_clean", 
                        regexp_replace(col("Pit_Time"), "[^0-9.]", "")) \
             .withColumn("race_norm", normalize_udf(col("Race Name"))) \
             .withColumn("driver_norm", normalize_udf(col("Driver"))) \
             .withColumn("pit_time_s", 
                        when(col("pit_time_clean") != "", col("pit_time_clean").cast(FloatType()))
                        .otherwise(None))

# Create a driver mapping from pit data (Driver -> normalized name)
if pit is not None:
    driver_mapping = pit.select(
        col("Driver").alias("driver_name"),
        col("driver_norm")
    ).distinct()

# For laps DataFrame, we need to join with driver mapping to get proper driver names
if laps is not None and 'driver_mapping' in locals():
    # First, let's check what driver information we have in laps
    print("Laps driverId sample:")
    laps.select("driverId").distinct().show(10)
    
    # Since laps only has driverId, not driver names, we need a different approach
    # Let's use the first few characters of driverId as a temporary match
    laps = laps.withColumn("driver_norm", col("driverId").cast(StringType()))

# Alternative approach: Let's check if we can match by race context instead
print("\n🔍 Checking data alignment...")

if pit is not None and laps is not None:
    # Check pit data structure
    print("Pit data sample:")
    pit.select("Race Name", "Driver", "driver_norm", "Pit_Lap").show(5, truncate=False)
    
    print("Laps data sample:")
    laps.select("raceId", "driverId", "driver_norm", "lap").show(5, truncate=False)
    
    # Since direct driver matching isn't working, let's try a different approach
    # We'll analyze pit stops without lap context for now
    print("\n📊 Analyzing pit stops without lap context (simplified analysis)...")
    
    # Basic pit stop analysis
    pit_analysis = pit.filter(col("Pit_Lap").isNotNull() & col("pit_time_s").isNotNull())
    
    # Group by race and driver to get pit stop statistics
    race_driver_stats = pit_analysis.groupBy("race_norm", "driver_norm").agg(
        avg("pit_time_s").alias("avg_pit_time"),
        count("pit_time_s").alias("pit_stop_count"),
        first("Circuit").alias("circuit"),
        first("Country").alias("country")
    )
    
    # Compare to race average
    race_avg = pit_analysis.groupBy("race_norm").agg(
        avg("pit_time_s").alias("race_avg_pit_time")
    )
    
    final_analysis = race_driver_stats.join(race_avg, "race_norm", "left") \
        .withColumn("performance_vs_avg", col("avg_pit_time") - col("race_avg_pit_time"))
    
    print("✅ Simplified pit stop analysis completed!")
    print(f"Analyzed {final_analysis.count()} driver-race combinations")
    
    # Show results
    print("\nTop 10 fastest pit stops (vs race average):")
    final_analysis.orderBy("performance_vs_avg").select(
        "race_norm", "driver_norm", "circuit", "avg_pit_time", "race_avg_pit_time", "performance_vs_avg"
    ).show(10, truncate=False)
    
    print("\nTop 10 slowest pit stops (vs race average):")
    final_analysis.orderBy(col("performance_vs_avg").desc()).select(
        "race_norm", "driver_norm", "circuit", "avg_pit_time", "race_avg_pit_time", "performance_vs_avg"
    ).show(10, truncate=False)

# If you need the lap context, we'll need to get proper driver mapping data
print("\n💡 To enable lap context analysis, you need:")
print("1. A driver lookup table mapping driverId to driver names")
print("2. Or add driver names to your laps data")
print("3. Or use a different join strategy based on race context")

# Save the cleaned pit data for later use
if pit is not None:
    cleaned_pit = pit.filter(col("Pit_Lap").isNotNull() & col("pit_time_s").isNotNull())
    print(f"\n💾 Cleaned pit data ready: {cleaned_pit.count()} rows")
    cleaned_pit.select("race_norm", "driver_norm", "Pit_Lap", "pit_time_s", "Circuit", "Country").show(5, truncate=False)

Laps driverId sample:


25/11/17 23:18:41 WARN TaskSetManager: Stage 151 contains a task of very large size (1626 KiB). The maximum recommended task size is 1000 KiB.


+--------+
|driverId|
+--------+
|     808|
|     155|
|     811|
|     822|
|      22|
|       1|
|      13|
|      16|
|       3|
|      20|
+--------+
only showing top 10 rows

🔍 Checking data alignment...
Pit data sample:
+---------------------+----------------------------+----------------------------+-------+
|Race Name            |Driver                      |driver_norm                 |Pit_Lap|
+---------------------+----------------------------+----------------------------+-------+
|Australian Grand Prix|Sebastian Vettel            |sebastian_vettel            |26.0   |
|Australian Grand Prix|Sebastian Vettel            |sebastian_vettel            |NaN    |
|Australian Grand Prix|Lewis Hamilton              |lewis_hamilton              |19.0   |
|Australian Grand Prix|Lewis Hamilton              |lewis_hamilton              |NaN    |
|Australian Grand Prix|Kimi R√É∆í√Ç¬§ikk√É∆í√Ç¬∂nen|kimi_r√é∆í√ç¬§ikk√é∆í√ç¬∂nen|18.0   |
+---------------------+----------------------------+--

Exception ignored in: <_io.BufferedWriter name=5>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 200, in manager
BrokenPipeError: [Errno 32] Broken pipe
25/11/17 23:18:41 WARN TaskSetManager: Stage 155 contains a task of very large size (1626 KiB). The maximum recommended task size is 1000 KiB.
Exception ignored in: <_io.BufferedWriter name=5>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 200, in manager
BrokenPipeError: [Errno 32] Broken pipe


Analyzed 841 driver-race combinations

Top 10 fastest pit stops (vs race average):
+------------------+----------------------+------------------------------------+------------------+-----------------+-------------------+
|race_norm         |driver_norm           |circuit                             |avg_pit_time      |race_avg_pit_time|performance_vs_avg |
+------------------+----------------------+------------------------------------+------------------+-----------------+-------------------+
|tuscan_grand_prix |esteban_ocon          |Autodromo Internazionale del Mugello|17.454999923706055|529.4695014953613|-512.0145015716553 |
|british_grand_prix|marcus_ericsson       |Silverstone Circuit                 |28.207000732421875|417.1504187912777|-388.9434180588558 |
|british_grand_prix|nyck_de_vries         |Silverstone Circuit                 |28.327500343322754|417.1504187912777|-388.8229184479549 |
|british_grand_prix|daniil_kvyat          |Silverstone Circuit                 |29.094000

In [35]:
final_lap = laps
if pit_joined is not None:
    pit_for_join = pit_joined.select("race_norm","driver_norm","pit_lap","pit_time_s","lap_time_before","lap_time_after","delta_after_vs_before_s","pit_time_norm")
    final_lap = final_lap.join(pit_for_join,(final_lap["lap"]==pit_for_join["pit_lap"]) & (final_lap["driver_norm"]==pit_for_join["driver_norm"]),"left")
if safety is not None:
    safety_small = safety.select("race_norm","Deployed","Retreated","FullLaps").dropDuplicates()
    final_lap = final_lap.join(safety_small,"race_norm","left")
if redflag is not None:
    red_small = redflag.select("race_norm","Lap","Incident").dropDuplicates()
    final_lap = final_lap.join(red_small,"race_norm","left")


In [49]:
# SIMPLE SOLUTION: Create new DataFrame with only the columns we actually need
if ANONYMIZE_DRIVERS and final_lap is not None:
    print("🔄 Creating clean DataFrame with essential columns only...")
    
    # Get the essential columns we want to keep
    essential_columns = [
        'race_norm', 'raceId', 'driverId', 'lap', 'position', 'time', 
        'milliseconds', 'lap_time_s', 'cum_time_s', 'gap_to_leader',
        'pos_inlap', 'gap_to_next', 'gap_to_prev', 'pit_lap', 'pit_time_s',
        'lap_time_before', 'lap_time_after', 'delta_after_vs_before_s', 
        'pit_time_norm'
    ]
    
    # Get the first driver_norm column data
    all_columns = final_lap.columns
    driver_norm_positions = [i for i, col_name in enumerate(all_columns) if "driver_norm" in col_name]
    
    if driver_norm_positions:
        first_pos = driver_norm_positions[0]
        
        # Create a simple list of rows with only the columns we want
        rows = final_lap.rdd.map(lambda row: (
            row[all_columns.index('race_norm')] if 'race_norm' in all_columns else None,
            row[all_columns.index('raceId')] if 'raceId' in all_columns else None,
            row[all_columns.index('driverId')] if 'driverId' in all_columns else None,
            row[all_columns.index('lap')] if 'lap' in all_columns else None,
            row[all_columns.index('position')] if 'position' in all_columns else None,
            row[all_columns.index('time')] if 'time' in all_columns else None,
            row[all_columns.index('milliseconds')] if 'milliseconds' in all_columns else None,
            row[all_columns.index('lap_time_s')] if 'lap_time_s' in all_columns else None,
            row[all_columns.index('cum_time_s')] if 'cum_time_s' in all_columns else None,
            row[all_columns.index('gap_to_leader')] if 'gap_to_leader' in all_columns else None,
            row[all_columns.index('pos_inlap')] if 'pos_inlap' in all_columns else None,
            row[all_columns.index('gap_to_next')] if 'gap_to_next' in all_columns else None,
            row[all_columns.index('gap_to_prev')] if 'gap_to_prev' in all_columns else None,
            row[all_columns.index('pit_lap')] if 'pit_lap' in all_columns else None,
            row[all_columns.index('pit_time_s')] if 'pit_time_s' in all_columns else None,
            row[all_columns.index('lap_time_before')] if 'lap_time_before' in all_columns else None,
            row[all_columns.index('lap_time_after')] if 'lap_time_after' in all_columns else None,
            row[all_columns.index('delta_after_vs_before_s')] if 'delta_after_vs_before_s' in all_columns else None,
            row[all_columns.index('pit_time_norm')] if 'pit_time_norm' in all_columns else None,
            row[first_pos]  # driver_norm data
        )).collect()
        
        # Create mapping for anonymization
        drivers = [row[-1] for row in rows if row[-1] is not None]
        unique_drivers = sorted(list(set(drivers)))
        mapping = {d: f"driver_{i+1:03d}" for i, d in enumerate(unique_drivers)}
        
        # Create final rows with anonymized driver
        final_rows = []
        for row in rows:
            original_driver = row[-1]
            anonymized_driver = mapping.get(original_driver, None)
            final_rows.append(row[:-1] + (anonymized_driver,))
        
        # Create new clean DataFrame
        from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, FloatType
        
        schema = StructType([
            StructField("race_norm", StringType(), True),
            StructField("raceId", IntegerType(), True),
            StructField("driverId", IntegerType(), True),
            StructField("lap", IntegerType(), True),
            StructField("position", IntegerType(), True),
            StructField("time", StringType(), True),
            StructField("milliseconds", IntegerType(), True),
            StructField("lap_time_s", FloatType(), True),
            StructField("cum_time_s", FloatType(), True),
            StructField("gap_to_leader", FloatType(), True),
            StructField("pos_inlap", IntegerType(), True),
            StructField("gap_to_next", FloatType(), True),
            StructField("gap_to_prev", FloatType(), True),
            StructField("pit_lap", IntegerType(), True),
            StructField("pit_time_s", FloatType(), True),
            StructField("lap_time_before", FloatType(), True),
            StructField("lap_time_after", FloatType(), True),
            StructField("delta_after_vs_before_s", FloatType(), True),
            StructField("pit_time_norm", FloatType(), True),
            StructField("driver", StringType(), True)
        ])
        
        final_lap = spark.createDataFrame(final_rows, schema)
        print(f"✅ Created clean DataFrame with {len(final_rows)} rows and anonymized drivers!")
        
    else:
        print("❌ No driver_norm columns found")
else:
    print("❌ No final_lap data available")

# Show the result
if final_lap is not None:
    print("Final schema:")
    final_lap.printSchema()
    print("Sample data:")
    final_lap.select("driver").distinct().show(10)
    print(f"Total rows: {final_lap.count()}")

🔄 Creating clean DataFrame with essential columns only...


✅ Created clean DataFrame with 589081 rows and anonymized drivers!
Final schema:
root
 |-- race_norm: string (nullable = true)
 |-- raceId: integer (nullable = true)
 |-- driverId: integer (nullable = true)
 |-- lap: integer (nullable = true)
 |-- position: integer (nullable = true)
 |-- time: string (nullable = true)
 |-- milliseconds: integer (nullable = true)
 |-- lap_time_s: float (nullable = true)
 |-- cum_time_s: float (nullable = true)
 |-- gap_to_leader: float (nullable = true)
 |-- pos_inlap: integer (nullable = true)
 |-- gap_to_next: float (nullable = true)
 |-- gap_to_prev: float (nullable = true)
 |-- pit_lap: integer (nullable = true)
 |-- pit_time_s: float (nullable = true)
 |-- lap_time_before: float (nullable = true)
 |-- lap_time_after: float (nullable = true)
 |-- delta_after_vs_before_s: float (nullable = true)
 |-- pit_time_norm: float (nullable = true)
 |-- driver: string (nullable = true)

Sample data:


25/11/17 23:32:13 WARN TaskSetManager: Stage 303 contains a task of very large size (4893 KiB). The maximum recommended task size is 1000 KiB.


+----------+
|    driver|
+----------+
|driver_063|
|driver_085|
|driver_053|
|driver_013|
|driver_129|
|driver_065|
|driver_011|
|driver_034|
|driver_046|
|driver_101|
+----------+
only showing top 10 rows
Total rows: 589081


25/11/17 23:32:13 WARN TaskSetManager: Stage 306 contains a task of very large size (4893 KiB). The maximum recommended task size is 1000 KiB.


In [50]:
keep_cols = ["race_norm","driver","lap","lap_time_s","cum_time_s","gap_to_leader","gap_to_next","gap_to_prev",
             "pit_time_s","lap_time_before","lap_time_after","delta_after_vs_before_s","pit_time_norm",
             "Deployed","Retreated","FullLaps","Lap","Incident"]
ml_table = final_lap.select([c for c in keep_cols if c in final_lap.columns]).orderBy("race_norm","driver","lap")
ml_table.write.mode("overwrite").parquet(OUTPUT_PARQUET)
ml_table.write.mode("overwrite").option("header",True).csv(OUTPUT_CSV)
print("[INFO] Full engineered dataset saved.")

25/11/17 23:33:24 WARN TaskSetManager: Stage 309 contains a task of very large size (4893 KiB). The maximum recommended task size is 1000 KiB.
25/11/17 23:33:25 WARN TaskSetManager: Stage 310 contains a task of very large size (4893 KiB). The maximum recommended task size is 1000 KiB.
25/11/17 23:33:25 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
25/11/17 23:33:25 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
25/11/17 23:33:25 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
25/11/17 23:33:26 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
25/11/17 23:33:26 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) o

[INFO] Full engineered dataset saved.


In [51]:
import os

# Check if files exist
print("📁 Checking saved files:")
print(f"Parquet location: {OUTPUT_PARQUET}")
print(f"CSV location: {OUTPUT_CSV}")

# Check if directories exist
parquet_exists = os.path.exists(OUTPUT_PARQUET)
csv_exists = os.path.exists(OUTPUT_CSV)

print(f"Parquet directory exists: {parquet_exists}")
print(f"CSV directory exists: {csv_exists}")

if parquet_exists:
    parquet_files = os.listdir(OUTPUT_PARQUET)
    print(f"Parquet files: {parquet_files}")

if csv_exists:
    csv_files = os.listdir(OUTPUT_CSV)
    print(f"CSV files: {csv_files}")

📁 Checking saved files:
Parquet location: output/f1_full_engineered.parquet
CSV location: output/f1_full_engineered_csv
Parquet directory exists: True
CSV directory exists: True
Parquet files: ['part-00009-023d2c53-65da-4f69-97bb-08dd6b4ac2a4-c000.snappy.parquet', '.part-00000-023d2c53-65da-4f69-97bb-08dd6b4ac2a4-c000.snappy.parquet.crc', 'part-00002-023d2c53-65da-4f69-97bb-08dd6b4ac2a4-c000.snappy.parquet', '.part-00003-023d2c53-65da-4f69-97bb-08dd6b4ac2a4-c000.snappy.parquet.crc', 'part-00005-023d2c53-65da-4f69-97bb-08dd6b4ac2a4-c000.snappy.parquet', '._SUCCESS.crc', '.part-00007-023d2c53-65da-4f69-97bb-08dd6b4ac2a4-c000.snappy.parquet.crc', 'part-00004-023d2c53-65da-4f69-97bb-08dd6b4ac2a4-c000.snappy.parquet', 'part-00008-023d2c53-65da-4f69-97bb-08dd6b4ac2a4-c000.snappy.parquet', '.part-00008-023d2c53-65da-4f69-97bb-08dd6b4ac2a4-c000.snappy.parquet.crc', '.part-00004-023d2c53-65da-4f69-97bb-08dd6b4ac2a4-c000.snappy.parquet.crc', 'part-00003-023d2c53-65da-4f69-97bb-08dd6b4ac2a4-c000.

In [52]:
# Convert to pandas and download as CSV
pandas_df = ml_table.limit(100000).toPandas()  # Limit if too large

# Download as CSV
pandas_df.to_csv('f1_engineered_data.csv', index=False)
print("✅ CSV file ready for download in your current directory")

# If you want the full dataset, download the parquet files
import shutil

# Copy parquet files to current directory
if os.path.exists(OUTPUT_PARQUET):
    shutil.copytree(OUTPUT_PARQUET, './f1_engineered_parquet', dirs_exist_ok=True)
    print("✅ Parquet files copied to ./f1_engineered_parquet/")

25/11/17 23:35:37 WARN TaskSetManager: Stage 317 contains a task of very large size (4893 KiB). The maximum recommended task size is 1000 KiB.


✅ CSV file ready for download in your current directory
✅ Parquet files copied to ./f1_engineered_parquet/
